In [ ]:
"""
Automated Clinical Notes Summarization (Aug 2024 - Oct 2024)
Tech: Python, Hugging Face Transformers, PyTorch, Datasets, BART/GPT, BioBERT

This single-file pipeline provides:
 - Data loading from CSV/TSV (columns: `note`, `summary`)
 - Tokenization and preprocessing
 - Optional BioBERT document embedding augmentation (adds a special token whose
   embedding is set to a projection of the BioBERT pooled embedding)
 - Fine-tuning BART (or other seq2seq models) using Hugging Face Trainer
 - Evaluation with ROUGE and BLEU
 - Inference script

Notes:
 - Designed to be practical and reproducible. Tweak hyperparams for your dataset.
 - Dependencies: transformers, datasets, evaluate, sacrebleu, torch, tqdm, sentencepiece(optional), pandas

Usage examples (shell):
 python automated_clinical_summarization.py --train_file data/train.csv --validation_file data/val.csv --output_dir outputs/bart_biobert

"""

import os
import argparse
from dataclasses import dataclass, field
from typing import Optional, Dict, Any

import torch
from torch import nn

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset, DatasetDict
import evaluate
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --------------------------- Utility / Config ---------------------------------
DEFAULT_MODEL = "facebook/bart-large"  # change to any seq2seq (e.g., "google/mt5-base")
BIOBERT_NAME = "dmis-lab/biobert-v1.1"

@dataclass
class Config:
    train_file: str
    validation_file: Optional[str]
    output_dir: str
    model_name_or_path: str = DEFAULT_MODEL
    biobert_augment: bool = True
    bio_token: str = "<bio>"
    max_input_length: int = 1024
    max_target_length: int = 128
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 4
    num_train_epochs: int = 3
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    fp16: bool = True
    seed: int = 42

# --------------------------- BioBERT augmentation -----------------------------
class BioAugmenter:
    """Compute pooled BioBERT embedding and project into seq2seq token embedding space.

    Implementation approach (practical & reproducible):
      - Tokenizer: add a new special token (e.g. <bio>) to the seq2seq tokenizer vocab
      - Resize the model embeddings
      - Use BioBERT to compute a pooled embedding for each input note (mean of last hidden)
      - Project that vector to the seq2seq embedding size and set it as the embedding for <bio>
      - This is a *feature-based* augmentation — the special token sits at beginning of input

    This approach avoids heavy architectural changes and is compatible with HF Trainer.
    """

    def __init__(self, biobert_name: str, device: str = "cpu"):
        self.device = device
        print(f"Loading BioBERT ({biobert_name}) on {device} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(biobert_name)
        self.model = AutoModel.from_pretrained(biobert_name).to(device)
        self.model.eval()

    def pooled_embedding(self, texts, batch_size=8):
        """Return numpy array (N x D) pooled embeddings for list of texts."""
        all_embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = self.tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=512).to(self.device)
            with torch.no_grad():
                out = self.model(**enc)
                # try pooler_output, otherwise mean of last_hidden_state
                pooled = None
                if hasattr(out, "pooler_output") and out.pooler_output is not None:
                    pooled = out.pooler_output
                else:
                    pooled = out.last_hidden_state.mean(dim=1)
                all_embs.append(pooled.cpu().numpy())
        return np.vstack(all_embs)

# --------------------------- Dataset helpers ---------------------------------

def load_csv_dataset(train_file, validation_file=None, text_col="note", summary_col="summary"):
    data_files = {"train": train_file}
    if validation_file:
        data_files["validation"] = validation_file
    ds = load_dataset("csv", data_files=data_files)
    # optionally cast columns if needed
    return ds

# --------------------------- Preprocessing -----------------------------------

def preprocess_function(examples, tokenizer, max_input_length, max_target_length, bio_token=None):
    inputs = examples["note"]
    # if bio_token present, prepend it to each input so model sees the embedding
    if bio_token:
        inputs = [bio_token + " " + x for x in inputs]
    model_inputs = tokenizer(inputs, max_length=max_input_length, padding="max_length", truncation=True)

    # Tokenize targets with the `text_target` keyword as per HF Trainer recommendations
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["summary"], max_length=max_target_length, padding="max_length", truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# --------------------------- Metric compute ----------------------------------

rouge = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

# sacrebleu prefers detokenized strings; for BLEU we will use sacrebleu via evaluate

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    # decode
    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    # Rouge
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    # rouge returns rouge1/2/l sum; keep common ones
    rouge_result = {k: round(v, 4) for k, v in result.items()}

    # BLEU (evaluate's BLEU expects tokenized references? It uses sacrebleu under the hood)
    # Provide references as list of list
    bleu = bleu_metric.compute(predictions=decoded_preds, references=[[r] for r in decoded_labels])

    out = {**rouge_result, "bleu": round(bleu["bleu"], 4)}
    return out

# --------------------------- Main pipeline -----------------------------------

def main(args: Config):
    global tokenizer  # used in compute_metrics

    # reproducibility
    torch.manual_seed(args.seed)

    # prepare dataset
    ds = load_csv_dataset(args.train_file, args.validation_file)

    # Initialize tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(args.model_name_or_path)

    # BioBERT augmentation: add special token and set embedding
    bio_augmenter = None
    if args.biobert_augment:
        # add token
        if args.bio_token in tokenizer.get_vocab():
            print(f"Token {args.bio_token} already in tokenizer.")
        else:
            tokenizer.add_tokens([args.bio_token])
            model.resize_token_embeddings(len(tokenizer))

        # load biobert
        device = "cuda" if torch.cuda.is_available() else "cpu"
        bio_augmenter = BioAugmenter(BIOBERT_NAME, device=device)

        # compute pooled embeddings for train+val and set embedding vector for bio_token
        # We'll compute a prototype pooled embedding by averaging a sample of training notes
        sample_texts = ds["train"]["note"]
        # sample up to 1024 notes for prototype
        sample_texts = sample_texts[: min(len(sample_texts), 1024)]
        pooled = bio_augmenter.pooled_embedding(sample_texts, batch_size=8)  # shape N x D
        proto = pooled.mean(axis=0)  # D

        # project proto -> model.embedding_size
        embedding_size = model.get_input_embeddings().weight.shape[1]
        proj = nn.Linear(proto.shape[0], embedding_size)
        with torch.no_grad():
            projected = torch.tensor(proto, dtype=torch.float32).unsqueeze(0)
            try:
                projected = proj(projected).squeeze(0)
            except Exception:
                # fallback to simple resize via padding/truncation
                projected = projected.squeeze(0)
                if projected.shape[0] > embedding_size:
                    projected = projected[:embedding_size]
                elif projected.shape[0] < embedding_size:
                    pad = torch.zeros(embedding_size - projected.shape[0], dtype=torch.float32)
                    projected = torch.cat([projected, pad], dim=0)

        # set the embedding for special token index
        bio_token_id = tokenizer.convert_tokens_to_ids(args.bio_token)
        with torch.no_grad():
            emb_weights = model.get_input_embeddings().weight
            emb_weights[bio_token_id, :] = projected.to(emb_weights.device)
        print(f"Set prototype BioBERT embedding to token id {bio_token_id} (" + args.bio_token + ")")

    # Preprocess datasets
    preprocess = lambda ex: preprocess_function(ex, tokenizer, args.max_input_length, args.max_target_length, bio_token=(args.bio_token if args.biobert_augment else None))
    tokenized = ds.map(preprocess, batched=True, remove_columns=ds["train"].column_names)

    # Data collator
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=args.output_dir,
        evaluation_strategy="epoch" if "validation" in tokenized else "no",
        per_device_train_batch_size=args.per_device_train_batch_size,
        per_device_eval_batch_size=args.per_device_eval_batch_size,
        num_train_epochs=args.num_train_epochs,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        fp16=args.fp16 and torch.cuda.is_available(),
        save_total_limit=3,
        seed=args.seed,
        predict_with_generate=True,
        logging_steps=50,
        push_to_hub=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized.get("validation", None),
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # Train
    trainer.train()

    # Evaluate
    if "validation" in tokenized:
        metrics = trainer.evaluate()
        print("Evaluation metrics:\n", metrics)

    # Save
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    # Optional: run a small inference demo on validation examples
    if "validation" in tokenized:
        samples = ds["validation"]["note"][:5]
        if args.biobert_augment:
            inputs = [args.bio_token + " " + s for s in samples]
        else:
            inputs = samples
        inputs_tok = tokenizer(inputs, return_tensors="pt", truncation=True, padding=True, max_length=args.max_input_length).to(model.device)
        with torch.no_grad():
            generated = model.generate(**inputs_tok, max_length=args.max_target_length, num_beams=2)
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        print("\n--- Sample predictions ---\n")
        for i, pred in enumerate(decoded):
            print(f"INPUT[{i}]: {samples[i][:200]}...\nPRED[{i}]: {pred}\n")

# --------------------------- CLI entrypoint ---------------------------------

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Clinical Notes Summarization Training Script")
    parser.add_argument("--train_file", type=str, required=True)
    parser.add_argument("--validation_file", type=str, default=None)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--model_name_or_path", type=str, default=DEFAULT_MODEL)
    parser.add_argument("--no_biobert", dest="biobert_augment", action="store_false")
    parser.add_argument("--max_input_length", type=int, default=1024)
    parser.add_argument("--max_target_length", type=int, default=128)
    parser.add_argument("--per_device_train_batch_size", type=int, default=2)
    parser.add_argument("--per_device_eval_batch_size", type=int, default=4)
    parser.add_argument("--num_train_epochs", type=int, default=3)
    parser.add_argument("--learning_rate", type=float, default=2e-5)
    parser.add_argument("--fp16", action="store_true")

    parsed = parser.parse_args()
    cfg = Config(
        train_file=parsed.train_file,
        validation_file=parsed.validation_file,
        output_dir=parsed.output_dir,
        model_name_or_path=parsed.model_name_or_path,
        biobert_augment=parsed.biobert_augment,
        max_input_length=parsed.max_input_length,
        max_target_length=parsed.max_target_length,
        per_device_train_batch_size=parsed.per_device_train_batch_size,
        per_device_eval_batch_size=parsed.per_device_eval_batch_size,
        num_train_epochs=parsed.num_train_epochs,
        learning_rate=parsed.learning_rate,
        fp16=parsed.fp16,
    )

    main(cfg)
